# 進階 Agent 範例：NVIDIA AI-Q Blueprint

在前面兩份 notebook 裡，我們從零開始建立了第一個 `react_agent` workflow、學會幫 agent 加上自訂工具與 RAG 檢索，並且在 `Custom_Agent_Workflows.ipynb` 親手用 LangGraph 寫了一個 `simple_router_agent`，把多個 sub-agent 編排成一個 workflow。

這份 notebook 要更進一步介紹：**[NVIDIA AI-Q Blueprint](https://github.com/NVIDIA-AI-Blueprints/aiq)** —— 一個建構在 NeMo Agent Toolkit 之上、企業等級的研究型 agent 範本。

AI-Q 把我們前面看到的 NAT 元件（YAML 設定、function、agent）組成了一個**多 agent、可路由、可深度推理**的研究助理：可以快速回答簡單問題（淺層研究），也能針對複雜題目產生具引用來源的長篇研究報告（深度研究）。

我們會分三段體會它：

1. **認識 AI-Q 架構**：看一張流程圖、核心元件、Workflow 編排器 `chat_deepresearcher_agent` 跟你寫的 `simple_router_agent` 怎麼對照（第 1 節）
2. **跑跑看 AI-Q**：用預設設定看看 meta / 淺層 / 深度三條路徑各長什麼樣（第 2–3 節）
3. **認識 AI-Q 的 Web UI 部署**：看一下 Web UI、API server、PostgreSQL 三件套的 Docker Compose 架構（第 4 節）

讀完之後，你會理解：當你已經習慣了 NAT 的「以 YAML 為核心」思維，怎麼以最小代價擴展成一個更大、可進入 production 的 agentic 系統。


## 目錄

- [0.0) 環境準備](#setup)
  - [0.1) 系統需求](#prereqs)
  - [0.2) API 金鑰](#api-keys)
  - [0.3) 取得 AI-Q Blueprint 原始碼](#clone-aiq)
  - [0.4) 安裝 AI-Q](#install-aiq)
- [1.0) 認識 AI-Q](#about-aiq)
  - [1.1) AI-Q 是什麼？](#what-is-aiq)
  - [1.2) 整體架構與工作流程](#architecture)
  - [1.3) 核心元件](#components)
  - [1.4) Workflow 編排器：`chat_deepresearcher_agent` 怎麼把 agent 串起來](#orchestrator)
- [2.0) 解讀設定檔](#config)
  - [2.1) 預設提供的設定檔](#preset-configs)
  - [2.2) 寫一份精簡版設定檔](#simple-config)
- [3.0) 執行 AI-Q](#run-aiq)
  - [3.1) 元資訊回應（Meta）](#meta)
  - [3.2) 淺層研究（Shallow Research）](#shallow)
  - [3.3) 深度研究（Deep Research）](#deep)
  - [3.4) Server 模式：用 `nat serve` 變成 HTTP API](#serve)
- [4.0) 認識 AI-Q 的 Web UI 部署架構](#deploy)
- [4.5) 動手 10 分鐘：打造你自己的 Agent 與工具](#build-your-own)
- [5.0) 下一步](#next-steps)

<span style="color:rgb(0, 31, 153); font-style: italic;">小提醒：如果你是在 Google Colab 上閱讀，建議切到左邊的「目錄」分頁來瀏覽，比較好導覽。</span>


<a id="setup"></a>
# 0.0) 環境準備

<a id="prereqs"></a>
## 0.1) 系統需求

本份 notebook 直接接續前兩份的環境往下教，所以我們假設：

- 你已經跑過 `adding_tools_to_agents.ipynb` 與 `composing_agents_into_workflows.ipynb`
- 你的 `NVIDIA_API_KEY` 已經設定好（在同一個 kernel 裡仍然有效）
- 你已經熟悉以下概念，這份 notebook **不會從頭再教一次**：
  - NAT 工具註冊（`@register_function` + `FunctionBaseConfig` + `FunctionInfo.from_fn`）
  - 「workflow 也是 NAT function」、`builder.get_function(...)`、`builder.get_llm(...)`
  - LangGraph 三件套：State、Node、Edge——`StateGraph`、`add_node`、`add_conditional_edges`、`compile`
  - `react_agent` 與 `tool_calling_agent` 的差別

額外需要的東西：

- **作業系統：** Linux 或 macOS（AI-Q 的 `setup.sh` 是 bash 腳本）
- **Python：** 3.11、3.12 或 3.13
- **git：** 用來把 AI-Q 原始碼抓下來
- **（可選）Docker / Docker Compose：** 如果你想體驗 Web UI 完整版部署
- **（可選）Node.js 22+：** 同上，Web UI 開發用

<span style="color:rgb(0, 31, 153); font-style: italic;"><b>小提醒：兩個 <code>.venv</code> 不會互相干擾</b><br>
AI-Q 的安裝腳本會在它自己的目錄底下另建一個 Python 3.13 的虛擬環境，跟你前面 notebook 用的環境是各自獨立的：

<ul>
<li>前兩份 notebook 用的：例如 <code>zh-TW/.venv/</code>（含 <code>nvidia-nat[langchain,llama-index]</code> 與 LangGraph）</li>
<li>本份 notebook 會建立的：<code>zh-TW/aiq/.venv/</code>（AI-Q 自己用的，包含完整的 AI-Q 依賴）</li>
</ul>

這是因為 cell 11 會 <code>%cd aiq</code>，把工作目錄切進 clone 下來的 <code>aiq/</code> 之後，<code>scripts/setup.sh</code> 才會在「當前目錄」建立 <code>.venv</code>。為了不要每次跑 <code>nat</code> 都要打 <code>.venv/bin/nat</code>，我們會在 0.4 節安裝完之後用 <code>os.environ</code> 把 <code>aiq/.venv/bin</code> 加到 kernel 的 <code>PATH</code> 最前面——這之後所有 <code>!nat ...</code> 指令就會精確走到 AI-Q 自己的 venv，不會跟前面 notebook 的 <code>nat</code> 打架。</span>


<a id="api-keys"></a>
## 0.2) API 金鑰

AI-Q 預設情境下會用到以下 API：

| API | 環境變數 | 用途 | 是否必填 |
| --- | --- | --- | --- |
| NVIDIA Build | `NVIDIA_API_KEY` | NIM 上的 LLM 推論 | 必填 |
| Tavily | `TAVILY_API_KEY` | Web Search 工具 | 強烈建議 |

若 `TAVILY_API_KEY` 沒設定，agent 仍會啟動，但「淺層研究」與「深度研究」會缺乏一個重要的資料來源（網路搜尋），請至少準備一個。

**取得方式：**

- **NVIDIA Build：** 到 [NVIDIA Build](https://build.nvidia.com) 註冊帳號，並在 https://build.nvidia.com/settings/api-keys 產生金鑰（如果你前一份 notebook 已輸入過，這裡就會自動沿用）。
- **Tavily：** 到 [tavily.com](https://tavily.com/) 註冊並產生 API key。

In [1]:
import getpass
import os

if "NVIDIA_API_KEY" not in os.environ or not os.environ["NVIDIA_API_KEY"]:
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("請輸入你的 NVIDIA API 金鑰：")

if "TAVILY_API_KEY" not in os.environ or not os.environ["TAVILY_API_KEY"]:
    os.environ["TAVILY_API_KEY"] = getpass.getpass("請輸入你的 Tavily API 金鑰（沒有的話可直接 Enter 跳過）：")

請輸入你的 NVIDIA API 金鑰： ········
請輸入你的 Tavily API 金鑰（沒有的話可直接 Enter 跳過）： ········


<a id="clone-aiq"></a>
## 0.3) 取得 AI-Q Blueprint 原始碼

AI-Q 是一個獨立的 Blueprint 專案，並非透過 `pip install` 就能拿到完整內容。我們會把整個 GitHub 專案 clone 下來，方便之後檢視原始碼、修改設定檔，以及執行隨附的腳本與 notebook。

In [2]:
%%bash
# 若已經 clone 過了就不用重複
if [ ! -d aiq ]; then
    git clone https://github.com/NVIDIA-AI-Blueprints/aiq.git
else
    echo "aiq 資料夾已存在，跳過 clone"
fi

Cloning into 'aiq'...


接下來把 notebook 的工作目錄切換到 AI-Q 專案根目錄，後續的相對路徑（`configs/`、`scripts/`、`deploy/` 等）就會跟官方文件一致。

In [3]:
%cd aiq

/localhome/local-clchiu/NeMo-Agent-Toolkit/examples/notebooks/zh-TW/aiq


<a id="install-aiq"></a>
## 0.4) 安裝 AI-Q

AI-Q 提供了一份一鍵安裝腳本 `scripts/setup.sh`，它會幫你完成：

1. 用 `uv` 建立一個 Python 3.13 的虛擬環境 `.venv/`
2. 安裝 AI-Q 核心套件與開發依賴（包含底層的 `nvidia-nat`）
3. 安裝 frontends：CLI、Debug Console、AI-Q API server
4. 安裝資料來源（Tavily、Google Scholar、Knowledge Layer 等）
5. 安裝 benchmarks（FreshQA、Deep Research Bench）
6. （若有 `npm`）安裝 Web UI 前端依賴

<span style="color:rgb(0, 31, 153); font-style: italic;">小提醒：第一次跑會花幾分鐘，因為要下載與編譯不少套件。</span>

In [4]:
!./scripts/setup.sh

=== AI-Q Blueprint Development Setup ===

uv is already installed

Creating virtual environment...
Using CPython 3.13.13
Creating virtual environment with seed packages at: .venv
 + pip==26.1.1
Activate with: source .venv/bin/activate
Virtual environment created

Activating virtual environment...

Installing core framework with dev dependencies...
Resolved 344 packages in 2ms
Prepared 8 packages in 830ms                                             
Uninstalled 1 package in 7ms
Installed 313 packages in 117ms                             
 + aioboto3==15.5.0
 + aiobotocore==2.25.1
 + aiofiles==25.1.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.4
 + aioitertools==0.13.0
 + aiorwlock==1.5.1
 + aiosignal==1.4.0
 + aiosqlite==0.22.1
 + aiq-agent==2.0.0 (from file:///localhome/local-clchiu/NeMo-Agent-Toolkit/examples/notebooks/zh-TW/aiq)
 + aiq-api==0.1.0 (from file:///localhome/local-clchiu/NeMo-Agent-Toolkit/examples/notebooks/zh-TW/aiq/frontends/aiq_api)
 + aiq-debug==0.1.0 (from file:///l

裝好之後，AI-Q 的 CLI 入口會是 `nat`。但有個小麻煩：在 Jupyter notebook 裡，**每個 `!` cell 都是獨立的 subshell**，所以 `!source .venv/bin/activate` 沒辦法跨 cell 生效——下一個 cell 開新 subshell 時 activate 又消失了。

改用 Python 直接修改 kernel 的 `PATH`，把 AI-Q venv 的 `bin/` 目錄加到最前面，效果跟 activate 一樣、而且**會跨所有後續 cell 持續**。設定完順便驗證一下安裝是否成功。

<span style="color:rgb(0, 31, 153); font-style: italic;">小提醒：這裡的 <code>nat</code> 本質上就是前面 notebook 用的同一個 NeMo Agent Toolkit CLI，只是裝在 AI-Q 自己的 venv 裡，並且註冊了 AI-Q 才有的 agent / function 型別（例如 <code>intent_classifier</code>、<code>shallow_research_agent</code>、<code>deep_research_agent</code>）。把 <code>aiq/.venv/bin</code> 放到 <code>PATH</code> 最前面，可以確保 <code>!nat</code> 解析到的就是 AI-Q 這個版本，而不是前面 notebook 那個。</span>


In [5]:
import os

os.environ["PATH"] = f"{os.getcwd()}/.venv/bin:" + os.environ["PATH"]

!nat --version

nat, version 1.6.0


<a id="about-aiq"></a>
# 1.0) 認識 AI-Q

<a id="what-is-aiq"></a>
## 1.1) AI-Q 是什麼？

AI-Q **研究型查詢** 的 AI 代理。

研究型查詢的特性是：

- 不只是一次工具呼叫就能回答，可能要做**多輪規劃、分節撰寫、彙整引用**
- 同時又不希望「我只問早安」也跑十分鐘的深度研究——所以系統要會**判斷意圖與深度**（這你在 `simple_router_agent` 的 `classify_node` 做過縮小版了）
- 回答要能**附上來源**，方便人去驗證

AI-Q 把這些需求拆成數個專責 agent，再用一個 [LangGraph](https://www.langchain.com/langgraph) 狀態機把它們串成一個 workflow。它的幾個關鍵特色：

- **Two-tier 研究架構**：簡單查詢走快速路線（淺層研究），複雜題目才進深度研究
- **意圖路由（Intent Routing）**：用一次 LLM 呼叫同時判斷「這是閒聊還是研究？」與「該走淺層還是深度？」（**升級版的 `classify_node`**）
- **可選的 Human-in-the-loop（HITL）**：深度研究前可以先讓使用者澄清題目、審核研究計畫
- **可組合**：每個 agent 都可以單獨跑，也可以組成完整 pipeline
- **以 YAML 為核心的設定**：跟 NAT 一致，agent / LLM / 工具 / 路由全部寫在設定檔，不必改程式碼
- **內建評測工具**：附 FreshQA、Deep Research Bench 等 benchmark 與 `nat eval` 整合
- **多種前端**：CLI、Web UI、SSE 串流的 async job API、Docker Compose 部署等


<a id="architecture"></a>
## 1.2) 整體架構與工作流程

AI-Q 的工作流程是一個 [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) `StateGraph`——**跟你 bridge notebook 親手蓋的那張用的是同一套 API**，只是節點數量、條件邊的複雜度上去了：

```mermaid
graph LR
    IC[Intent Classifier] -->|meta| END_NODE[END]
    IC -->|research / shallow| SR[Shallow Researcher]
    IC -->|research / deep| CL[Clarifier]
    SR -->|escalate| CL
    SR -->|done| END_NODE
    CL --> DR[Deep Researcher]
    DR --> END_NODE

    style IC fill:#fff3e0
    style SR fill:#f3e5f5
    style CL fill:#fce4ec
    style DR fill:#e8eaf6
```

**路由邏輯一句話總結：**

1. 所有 query 一律先進入 **Intent Classifier**（你的 `classify_node` 的 AI-Q 等價物）
2. 如果是 meta（打招呼、問你會做什麼…）→ 立刻產生回覆，結束
3. 如果是 research：
    - 簡單 → 進 **Shallow Researcher** 直接做工具呼叫
    - 複雜 → 先進 **Clarifier**，視情況跟使用者澄清題目、批准研究計畫，再交給 **Deep Researcher** 做多輪研究
4. 淺層研究若發現自己回答不了（escalate），也會回到 Clarifier、再進深度研究（**這個是你 bridge 沒做過的：從一個非起點節點再導向另一個節點**）

整個過程的中介狀態（對話歷史、意圖、深度、研究結果、引用…）都由一個叫 `ChatResearcherState` 的 Pydantic 物件帶在身上——**就是你寫 `RouterState` 的同一個套路**，只是欄位多很多（1.4 節會做欄位對照表）。


<a id="components"></a>
## 1.3) 核心元件

下表把每個元件對應到的 NAT `_type` 與職責整理一下，幫你之後讀設定檔時對得起來：

| 元件 | `_type` | 職責 |
| --- | --- | --- |
| **Intent Classifier** | `intent_classifier` | 一次 LLM 呼叫同時判斷「意圖（meta / research）」與「深度（shallow / deep）」，並在 meta 情境直接給回覆 |
| **Clarifier Agent** | `clarifier_agent` | 進入深度研究前，視情況跟使用者澄清題意、產生研究計畫並請使用者批准（HITL，可關閉） |
| **Shallow Researcher** | `shallow_research_agent` | 受工具呼叫次數限制的快路徑，適合單步驟查詢、簡單比較 |
| **Deep Researcher** | `deep_research_agent` | 多階段研究：orchestrator 統籌、planner 規劃章節、researcher 蒐證，最終彙整成帶引用的長篇報告 |
| **Chat Researcher Orchestrator** | `chat_deepresearcher_agent` | LangGraph 狀態機，串起上述所有 agent，作為 workflow 進入點 |

其中 **Deep Researcher** 自己內部又是一個小型的多 agent 系統，使用 [`deepagents`](https://docs.langchain.com/oss/python/deepagents/overview) 套件協調：

- **Orchestrator** 負責主導整個流程、彙整草稿、整理引用
- **planner-agent** 透過搜尋逐步建立帶證據的章節大綱
- **researcher-agent** 真正去執行各段研究、做內容彙整

這些 subagent 都會吃工具（最主要是 web search），跑完之後 orchestrator 把結果整成一份完整報告。

如果想看細節，AI-Q 在 `docs/source/architecture/agents/` 底下對每個元件都有獨立文件，例如：
- `intent-classifier.md`
- `clarifier.md`
- `shallow-researcher.md`
- `deep-researcher.md`

<a id="orchestrator"></a>
## 1.4) Workflow 編排器：`chat_deepresearcher_agent`

還記得你在 `composing_agents_into_workflows.ipynb` 寫的 `simple_router_agent` 嗎？AI-Q 的 `chat_deepresearcher_agent` 就是它的「**進階版**」——**同樣的骨架，只是 sub-agent 變多、路由變複雜、再加上 checkpoint 與 HITL**。

直接拿三個 workflow 並排比較：

| | `react_agent` | `simple_router_agent`（你寫的）| `chat_deepresearcher_agent`（AI-Q）|
| --- | --- | --- | --- |
| **本質** | 單一 agent（while 迴圈） | LangGraph router | LangGraph orchestrator |
| **sub-agent 數量** | 0 | 2（weather、blog） | 4（intent、shallow、deep、clarifier） |
| **內部結構** | 1 顆 LLM + N 個工具 | 5 個 node、1 條條件邊 | 4 個 node、**2 條條件邊** + escalation |
| **State 物件** | 框架自帶 | `RouterState`（3 個欄位） | `ChatResearcherState`（10+ 個欄位） |
| **checkpoint** | 沒有 | 沒有 | 有（SQLite / Postgres） |
| **HITL（暫停問人）** | 沒有 | 沒有 | 有（Clarifier 節點透過 `user_interaction_manager` 互動） |
| **怎麼拿 sub-agent？** | N/A | `builder.get_function(...)` | **完全一樣** |
| **怎麼蓋圖？** | N/A | `StateGraph(...).add_node().add_conditional_edges().compile()` | **完全一樣** |

**白話文**：你已經會寫 70% 的 `chat_deepresearcher_agent` 了。

### 1.4.1) 跟你寫的 `_build_graph()` 對照

打開 AI-Q 的 `aiq/src/aiq_agent/agents/chat_researcher/agent.py`、跳到 369 行。下面這段你**已經寫過了**——只是當時你寫的是 `RouterState`、5 個節點、1 條條件邊：

```369:399:/localhome/local-clchiu/aiq/src/aiq_agent/agents/chat_researcher/agent.py
        graph = StateGraph(ChatResearcherState)

        graph.add_node("intent_classifier", intent_classifier_node)
        graph.add_node("shallow_research", shallow_research_node)
        graph.add_node("clarifier", clarifier_node)
        graph.add_node("deep_research", deep_research_node)

        graph.set_entry_point("intent_classifier")

        graph.add_conditional_edges(
            "intent_classifier",
            route_after_orchestration,
            {
                "END": END,
                "clarifier": "clarifier",
                "shallow_research": "shallow_research",
            },
        )

        graph.add_conditional_edges(
            "shallow_research",
            should_escalate,
            {
                "deep_research": "clarifier",
                END: END,
            },
        )

        graph.add_edge("deep_research", END)

        return graph.compile(checkpointer=self.checkpointer)
```

跟你在 `simple_router_workflow.py` 寫的並排：

```python
# 你的：                                     # AI-Q 的：
graph = StateGraph(RouterState)              graph = StateGraph(ChatResearcherState)
graph.add_node("classify", classify_node)    graph.add_node("intent_classifier", ...)
graph.add_node("weather", weather_node)      graph.add_node("shallow_research", ...)
graph.add_node("blog", blog_node)            graph.add_node("clarifier", ...)
graph.add_node("meta", meta_node)            graph.add_node("deep_research", ...)
graph.add_node("unknown", unknown_node)
graph.set_entry_point("classify")            graph.set_entry_point("intent_classifier")
graph.add_conditional_edges(                 graph.add_conditional_edges(
    "classify", route_by_intent, {...}           "intent_classifier", route_after_orchestration, {...}
)                                            )
                                             graph.add_conditional_edges(  # ← 多一條！
                                                 "shallow_research", should_escalate, {...}
                                             )
for t in (...): graph.add_edge(t, END)       graph.add_edge("deep_research", END)
compiled_graph = graph.compile()             return graph.compile(checkpointer=...)
```

**API 完全一樣**——`StateGraph`、`add_node`、`set_entry_point`、`add_conditional_edges`、`compile`，你都用過了。

至於 sub-agent 是怎麼被拉進來的，也是熟悉招式：`register.py` 第 207-211 行裡的 `builder.get_function("intent_classifier")`、`builder.get_function("shallow_research_agent")` 等等——跟你 `await builder.get_function(config.weather_agent)` 一模一樣，這就是為什麼 YAML 的 `functions:` 區塊要先把每個 sub-agent 註冊好。


### 1.4.2) AI-Q 額外多帶進來的三樣東西

骨架你已經會了。剩下這三個是 AI-Q 比你的 simple_router_agent 多出來的，**值得專門認識**：

#### 1. `checkpointer`：對話可以恢復

AI-Q 在 `compile()` 時傳了一個 `checkpointer`，把 LangGraph 的 state 寫進 SQLite/Postgres：

- 同一 `conversation_id` 的第 2 句訊息進來時，state 會被讀回來——也就是 **AI-Q 天然支援多輪對話**
- 跑到一半當機了，重啟還能從上次斷點繼續

你的 simple_router_agent 沒帶 checkpointer，所以每次跑都是新對話。**單次 query 沒差**，但要做客服那種多輪互動就會需要。

#### 2. `ChatResearcherState`：更豐富的「行李」

| 你的 `RouterState`（3 個欄位） | AI-Q 的 `ChatResearcherState`（節錄） |
| --- | --- |
| `query: str` | `messages: list[BaseMessage]`（**完整對話歷史**，用 LangGraph reducer 自動 append） |
| `intent: str \| None` | `user_intent: UserIntent`（含 `intent` 與 `confidence`） |
| `answer: str \| None` | `depth_decision: DepthDecision`（meta / shallow / deep + 理由） |
| | `shallow_result: ShallowResult`（含 `escalate_to_deep` 旗標） |
| | `clarifier_result: str`（澄清對話的紀錄） |
| | `available_documents: list[...]`（使用者上傳的文件摘要） |
| | `data_sources: list[str]`（要篩哪些來源） |
| | ... |

**state 攜帶的資料越多，下游節點就能做越多事**——但維護成本也越高。AI-Q 把它包成一個 Pydantic 物件來管，剛好就是 LangGraph `StateGraph(...)` 第一行的型別參數，跟你用 `RouterState` 一模一樣。

#### 3. Escalation：條件邊的進階用法

從 `shallow_research` 出來那條條件邊，是你 bridge 沒寫過的進階模式：

```python
graph.add_conditional_edges(
    "shallow_research", should_escalate,
    {
        "deep_research": "clarifier",  # ← 標籤跟下一站不一樣！
        END: END,
    },
)
```

**注意這個小細節**：函式 `should_escalate` 回傳的標籤是 `"deep_research"`，但 mapping dict 把它導向 `"clarifier"`（先繞道澄清）。這是 LangGraph 的常見技巧——**回傳標籤代表「意圖」，實際下一站靠 dict 決定**。

在你寫的 `route_by_intent` 裡，標籤跟下一站剛好同名（`"weather"` → `"weather"`、`"blog"` → `"blog"`），所以你還沒看到這個技巧的價值。AI-Q 這裡示範了一個經典用例：「**邏輯上『要升級到深度研究』；但物理上要先繞到 Clarifier 確認題目**」——把這層 indirection 用 mapping dict 一句話搞定，比寫 if/else 漂亮太多。


### 1.4.3) 接下來看實際的 YAML

理論看完了。**接下來 2.0 節我們會直接看 AI-Q 的 `config_simple_researcher.yml`**——你會看到 1.3 表格裡每一個元件都在 `functions:` 區塊裡乖乖地註冊好，而 `workflow:` 區塊就指定 `_type: chat_deepresearcher_agent`，把它們全部串起來。

**形狀跟你 bridge notebook 的 `config.yml` 一模一樣**，只是：

- `llms:` 區塊多了幾顆 LLM（不同階段用不同模型，後面會解釋）
- `functions:` 區塊多了幾個 sub-agent（intent、shallow、deep）
- `workflow:` 區塊指向 `chat_deepresearcher_agent` 而非 `simple_router_agent`


<a id="config"></a>
# 2.0) 解讀設定檔

<a id="preset-configs"></a>
## 2.1) 預設提供的設定檔

AI-Q 在 `configs/` 目錄底下準備了好幾份**現成可用**的 YAML，讓你依照部署情境挑一個：

| 設定檔 | 用途 |
| --- | --- |
| `config_cli_default.yml` | CLI 預設：啟用 HITL 澄清與計畫核可、Web Search、（可選）論文搜尋 |
| `config_web_default_llamaindex.yml` | Web UI 預設：含 LlamaIndex Knowledge Layer 上傳檔案做 RAG |
| `config_web_frag.yml` | Web + Foundational RAG（外部 RAG server，Helm 部署用） |
| `config_frontier_models.yml` | 改用前沿模型（如 GPT-5.2）做 orchestrator/planner，researcher 仍走開源模型 |

我們先看一下目錄裡到底有哪些：

In [6]:
!ls configs/

config_cli_default.yml	    config_web_default_llamaindex.yml
config_frontier_models.yml  config_web_frag.yml
config_skills.yml


<a id="simple-config"></a>
## 2.2) 寫一份精簡版設定檔

為了在 notebook 裡跑得快、好懂，我們不直接使用 `config_cli_default.yml`，而是仿造它寫一份**精簡版**，把 HITL 部分先關掉（HITL 在 Jupyter 裡互動體驗較差），讓我們可以一個指令就跑完。

In [7]:
%%writefile config_simple_researcher.yml
general:
  telemetry:
    logging:
      console:
        _type: console
        level: INFO

llms:
  nemotron_llm_intent:
    _type: nim
    model_name: nvidia/nemotron-3-nano-30b-a3b
    base_url: "https://integrate.api.nvidia.com/v1"
    temperature: 0.5
    top_p: 0.9
    max_tokens: 4096
    num_retries: 5
    chat_template_kwargs:
      enable_thinking: true

  nemotron_nano_llm:
    _type: nim
    model_name: nvidia/nemotron-3-nano-30b-a3b
    base_url: "https://integrate.api.nvidia.com/v1"
    temperature: 0.1
    top_p: 0.3
    max_tokens: 16384
    num_retries: 5
    chat_template_kwargs:
      enable_thinking: true

  gpt_oss_llm:
    _type: nim
    model_name: openai/gpt-oss-120b
    base_url: https://integrate.api.nvidia.com/v1
    temperature: 1.0
    top_p: 1.0
    max_tokens: 256000
    api_key: ${NVIDIA_API_KEY}
    max_retries: 10

functions:
  web_search_tool:
    _type: tavily_web_search
    max_results: 5
    max_content_length: 1000

  advanced_web_search_tool:
    _type: tavily_web_search
    max_results: 2
    advanced_search: true

  intent_classifier:
    _type: intent_classifier
    llm: nemotron_llm_intent
    tools:
      - web_search_tool

  shallow_research_agent:
    _type: shallow_research_agent
    llm: nemotron_nano_llm
    tools:
      - web_search_tool

  deep_research_agent:
    _type: deep_research_agent
    orchestrator_llm: gpt_oss_llm
    planner_llm: gpt_oss_llm
    researcher_llm: nemotron_nano_llm
    max_loops: 2
    tools:
      - advanced_web_search_tool

workflow:
  _type: chat_deepresearcher_agent
  interactive_auth: false
  checkpoint_db: ${AIQ_CHECKPOINT_DB:-./checkpoints.db}

Writing config_simple_researcher.yml


In [8]:
"""
切換 web search 後端：從 Tavily（需 key）→ DuckDuckGo（keyless）。

整段做四件事：
  1. 在 sources/ddg_web_search/ 寫出一個 NAT plugin（跟 aiq/sources/tavily_web_search 同構，但底層改用 ddgs）
  2. pip install -e 進 AI-Q 的 venv（會順手裝 ddgs；靠 cell 15 設好的 PATH，!pip 會命中 aiq/.venv/bin/pip）
  3. 把 config_simple_researcher.yml 裡的 _type: tavily_web_search 換成 _type: ddg_web_search
  4. 在 YAML 裡補一個 data_source_registry，否則 shallow / deep research 會以「沒有來源」收場
     (詳見 aiq/src/aiq_agent/common/citation_verification.py 與 data_source_registry.py)

跑完之後直接重跑 3.x 節的 nat run，不再需要 TAVILY_API_KEY。
想退回 Tavily：把 cfg.write_text(...) 那段註解掉、重跑 Cell 30 即可。
這個 cell 是 idempotent 的——重複執行不會把 data_sources 重複插入。

提醒：DDG 是 keyless 但對 burst 連續查詢有 rate limit，shallow 沒事、deep research 偶爾會撞牆，
重跑通常可以恢復；要更穩可以調高 _retry_delay。
"""
from pathlib import Path

# ---- 1. 寫出 plugin ----------------------------------------------------------
P = Path("sources/ddg_web_search")
(P / "src").mkdir(parents=True, exist_ok=True)

(P / "pyproject.toml").write_text("""\
[build-system]
build-backend = "setuptools.build_meta"
requires = ["setuptools >= 64"]

[tool.setuptools]
packages = ["ddg_web_search"]
package-dir = {"ddg_web_search" = "src"}

[project]
name = "ddg-web-search"
version = "1.0.0"
requires-python = ">=3.11,<3.14"
dependencies = ["ddgs>=9.0"]

[project.entry-points."nat.plugins"]
ddg_web_search = "ddg_web_search.register"
""")

(P / "src/__init__.py").write_text("")

(P / "src/register.py").write_text(r'''
"""Keyless web search via DuckDuckGo (the `ddgs` package).
Drop-in replacement for `tavily_web_search` in AI-Q's shallow / deep researcher."""
import asyncio
import logging
from pydantic import ConfigDict, Field

from nat.builder.builder import Builder
from nat.builder.function_info import FunctionInfo
from nat.cli.register_workflow import register_function
from nat.data_models.function import FunctionBaseConfig

logger = logging.getLogger(__name__)


class DDGWebSearchConfig(FunctionBaseConfig, name="ddg_web_search"):
    # 容忍 YAML 裡留下來的 Tavily 旋鈕（advanced_search 之類），不會報錯
    model_config = ConfigDict(extra="ignore")
    max_results: int = Field(default=5, description="Max number of search results.")
    max_content_length: int | None = Field(
        default=1000,
        description="Truncate each result body to N chars to save downstream tokens.",
    )
    region: str = Field(default="wt-wt", description="DDG region (e.g. wt-wt = worldwide, us-en, tw-tzh).")
    safesearch: str = Field(default="moderate", description="off / moderate / strict")
    timeout_seconds: float = Field(default=15.0)
    max_retries: int = Field(default=2, description="Retry transient errors (rate-limit / timeout).")
    retry_delay_seconds: float = Field(default=2.0, description="Initial backoff between retries.")


def _search_sync(query, region, safesearch, max_results):
    from ddgs import DDGS
    with DDGS() as ddgs:
        return list(ddgs.text(
            query,
            region=region,
            safesearch=safesearch,
            max_results=max_results,
        ))


@register_function(config_type=DDGWebSearchConfig)
async def ddg_web_search(config: DDGWebSearchConfig, _builder: Builder):

    def _truncate(text: str) -> str:
        if config.max_content_length and len(text) > config.max_content_length:
            return text[: config.max_content_length - 3] + "..."
        return text

    async def _search(question: str) -> str:
        """Retrieves relevant contexts from web search (DuckDuckGo, keyless) for the given question.

        Args:
            question (str): The question to be answered.

        Returns:
            str: Up to N matching web pages, formatted as <Document> blocks with title, URL, and snippet.
        """
        if len(question) > 400:
            question = question[:397] + "..."

        last_err = None
        for attempt in range(config.max_retries + 1):
            try:
                results = await asyncio.wait_for(
                    asyncio.to_thread(
                        _search_sync, question, config.region, config.safesearch, config.max_results,
                    ),
                    timeout=config.timeout_seconds,
                )
                break
            except asyncio.TimeoutError as exc:
                last_err = f"timeout after {config.timeout_seconds}s"
                logger.warning("DDG (attempt %d) timed out", attempt + 1)
            except Exception as exc:
                last_err = str(exc)
                logger.warning("DDG (attempt %d) failed: %s", attempt + 1, exc)
            if attempt < config.max_retries:
                await asyncio.sleep(config.retry_delay_seconds * (2 ** attempt))
        else:
            return f"Error: DuckDuckGo search failed after {config.max_retries + 1} attempts ({last_err})."

        if not results:
            return f"No DuckDuckGo results for: {question}"

        # Match Tavily's <Document href="..."> output shape so AI-Q's citation parser handles us identically.
        docs = "\n\n---\n\n".join(
            f'<Document href="{r.get("href", "")}">\n<title>\n{r.get("title", "")}\n</title>\n{_truncate(r.get("body") or "")}\n</Document>'
            for r in results
        )
        return docs

    yield FunctionInfo.from_fn(_search, description=_search.__doc__)
''')

# ---- 2. 安裝（順手裝 ddgs 套件）-----------------------------------------------
!pip install -e sources/ddg_web_search --quiet

# ---- 3. Patch YAML (idempotent) ---------------------------------------------
cfg = Path("config_simple_researcher.yml")
text = cfg.read_text()

# 3a. tavily / wikipedia → ddg（兩個都換，重複跑也 OK）
text = text.replace("_type: tavily_web_search", "_type: ddg_web_search")
text = text.replace("_type: wikipedia_search", "_type: ddg_web_search")

# 3b. 補上 data_source_registry —— 否則 source registry 不會接住工具回的 URL
DATA_SOURCES_BLOCK = """  data_sources:
    _type: data_source_registry
    sources:
      - id: web_search
        name: "Web Search"
        description: "Search the web via DuckDuckGo (keyless)."
        tools:
          - web_search_tool
          - advanced_web_search_tool

"""
if "data_source_registry" not in text:
    text = text.replace("  intent_classifier:", DATA_SOURCES_BLOCK + "  intent_classifier:")

cfg.write_text(text)

print("✓ ddg_web_search plugin 已安裝")
print("✓ config_simple_researcher.yml 已切換成 DuckDuckGo + data_source_registry")
print("  現在直接跑下面 3.x 節的 nat run cells 即可，不需要 TAVILY_API_KEY。")

✓ ddg_web_search plugin 已安裝
✓ config_simple_researcher.yml 已切換成 DuckDuckGo + data_source_registry
  現在直接重跑下面 3.x 節的 nat run cells 即可，不需要 TAVILY_API_KEY。


對照 bridge notebook 那份 `config.yml`，這份 YAML 的每一塊都應該很眼熟——只是規模放大了：

- **`general`** —— 一般設定。這裡只開啟 console log。
- **`llms`** —— 三個不同設定的 NIM LLM。bridge 你只用 1 顆 `nim_llm`；AI-Q 把「給 classifier 用 / 給 researcher 用 / 給 orchestrator 用」**分到不同 LLM**：
    - `nemotron_llm_intent`：給 Intent Classifier 用（溫度較高一點，方便判讀意圖）
    - `nemotron_nano_llm`：給 Shallow Researcher 與 Deep 的 researcher subagent 用
    - `gpt_oss_llm`：給 Deep Researcher 的 orchestrator / planner 用，需要較長上下文視窗
    
    這也是「**workflow function 內可以呼叫多顆 LLM**」的好處——你 bridge 的 `simple_router_agent` 用同一顆 LLM 做 classify 跟跑 sub-agent；AI-Q 則為不同任務挑不同模型。
- **`functions`** —— 把工具與 agent 都當成 function 註冊起來：
    - `web_search_tool` / `advanced_web_search_tool` 是 Tavily Web Search 兩種不同設定
    - `intent_classifier`、`shallow_research_agent`、`deep_research_agent` 分別綁定它們專用的 LLM 與工具
    
    這層結構跟你 bridge 的 `weather_agent` / `blog_agent` **完全一樣**——sub-agent 在 `functions:` 區塊裡先「準備好」，workflow 在啟動時 `builder.get_function(...)` 把它們抓出來。
- **`workflow`** —— 進入點是 `chat_deepresearcher_agent`，也就是 1.2 節那張流程圖背後的 LangGraph 狀態機。

為了在 notebook 跑得單純，我們把 `interactive_auth` 設為 `false`，並且**刻意不**設定 `clarifier_agent` 與 `enable_clarifier`——這代表深度研究會直接開跑、不問澄清問題。等你之後想體驗 HITL，可以改用 `configs/config_cli_default.yml` 或自己加上 `clarifier_agent`。


<a id="run-aiq"></a>
# 3.0) 執行 AI-Q

終於可以開始跑了。AI-Q 的執行方式跟前面 notebook 完全一致——用 NAT CLI 的 `nat run` 指令；我們在 0.4 節已經把 `aiq/.venv/bin` 加到 `PATH`，所以**直接打 `nat` 就會用到 AI-Q 自己的 venv**：

```bash
nat run --config_file config_simple_researcher.yml --input "你的問題"
```

接下來我們會用同一份設定檔、針對三種典型輸入觀察 workflow 的不同路徑。


<a id="meta"></a>
## 3.1) 元資訊回應（Meta）

先丟一個明顯是「閒聊 / 自我介紹」的問題，看看 Intent Classifier 會不會走 meta 那條路。

**預期結果：** workflow 在 Intent Classifier 那一步直接結束，**完全不會**呼叫 web search 工具，更不會跑深度研究。

In [9]:
!nat run --config_file config_simple_researcher.yml --input "你好，請問你能幫我做什麼？"

/localhome/local-clchiu/NeMo-Agent-Toolkit/examples/notebooks/zh-TW/aiq/.venv/lib/python3.13/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
2026-05-19 02:21:04 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'config_simple_researcher.yml'
2026-05-19 02:21:04 - INFO     - aiq_agent.common.data_source_registry:169 - Loaded 1 data source(s): 2 tool(s), 0 group(s)
/localhome/local-clchiu/.local/share/uv/python/cpython-3.13.13-linux-x86_64-gnu/lib/python3.13/contextlib.py:214: UserWarning: WARNING! chat_template_kwargs is not default parameter.
                chat_template_kwargs was transferred to model_kwargs.
                Please confirm that chat_template_kwargs is wha

在 log 裡你應該會看到：

1. `intent_classifier` 被呼叫，輸出 `intent=meta`
2. workflow 直接結束，回傳一段歡迎訊息（內容會由 LLM 即時產生）

這正是「two-tier 路由」省下成本的地方：閒聊問題不會浪費任何 web search quota。

<a id="shallow"></a>
## 3.2) 淺層研究（Shallow Research）

下一題是一個**單一事實型**問題，預期 Intent Classifier 會判定為 research / shallow，於是會由 `shallow_research_agent` 接手、用 Tavily 做一兩次工具呼叫之後給出帶引用的回答。

<span style="color:rgb(0, 31, 153); font-style: italic;">小提醒：這一題需要 <code>TAVILY_API_KEY</code> 才能完整跑完。若沒設定，agent 仍會嘗試回答，但會缺乏網路上的最新資料。</span>

In [10]:
!nat run --config_file config_simple_researcher.yml --input "NVIDIA Spectrum-X 主要是設計來做什麼用的？"

/localhome/local-clchiu/NeMo-Agent-Toolkit/examples/notebooks/zh-TW/aiq/.venv/lib/python3.13/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
2026-05-19 02:21:16 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'config_simple_researcher.yml'
2026-05-19 02:21:16 - INFO     - aiq_agent.common.data_source_registry:169 - Loaded 1 data source(s): 2 tool(s), 0 group(s)
/localhome/local-clchiu/.local/share/uv/python/cpython-3.13.13-linux-x86_64-gnu/lib/python3.13/contextlib.py:214: UserWarning: WARNING! chat_template_kwargs is not default parameter.
                chat_template_kwargs was transferred to model_kwargs.
                Please confirm that chat_template_kwargs is wha

在 log 裡你應該會看到：

1. `shallow_research_agent` 啟動，做了一輪以上的 `web_search_tool` 呼叫
2. 最後 LLM 把搜尋結果彙整成一段帶來源連結的答案

如果你回去比對前面 notebook 寫的 `react_agent`，會發現結構非常像——只是這裡多了「事先判斷意圖、限制工具呼叫次數、強制最後一輪做 synthesis」這些設計。

<a id="deep"></a>
## 3.3) 深度研究（Deep Research）

最後我們丟一題明顯**需要多章節報告**的問題。Intent Classifier 應該會判定為 research / deep，並把工作交給 Deep Researcher。

<div class="alert alert-block alert-info">
<b>注意：</b>深度研究會跑好幾分鐘——它會經歷「規劃 → 多輪研究 → 草稿 → 引用整理 → 最終報告」這些階段，期間會做大量 LLM 呼叫與 web search。請耐心等候，並觀察 log 看每個 subagent（planner、researcher、orchestrator）在做什麼。
</div>

In [23]:
#!nat run --config_file config_simple_researcher.yml --input "請針對 NVIDIA GB200 與 NVIDIA B200 兩個 AI 加速器平台做一份完整的技術比較"

/localhome/local-clchiu/NeMo-Agent-Toolkit/examples/notebooks/zh-TW/aiq/.venv/lib/python3.13/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
2026-05-18 15:55:42 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'config_simple_researcher.yml'
2026-05-18 15:55:42 - INFO     - aiq_agent.common.data_source_registry:169 - Loaded 1 data source(s): 2 tool(s), 0 group(s)
/localhome/local-clchiu/.local/share/uv/python/cpython-3.13.13-linux-x86_64-gnu/lib/python3.13/contextlib.py:214: UserWarning: WARNING! chat_template_kwargs is not default parameter.
                chat_template_kwargs was transferred to model_kwargs.
                Please confirm that chat_template_kwargs is wha

跑完之後，你會拿到一份分章節、有引用編號的長篇報告。值得停下來想一下這個對比：

- 你 bridge notebook 寫的 `simple_router_agent`，**只能在 weather / blog / meta / unknown 之間挑一個分支跑**——一次 LLM 呼叫做 classify、一次 sub-agent 呼叫拿答案。
- AI-Q 這份 deep query 跑下來：意圖分類 → 規劃章節大綱 → planner 分頭蒐證 → orchestrator 彙整引用 → 產生 markdown 報告。整個 pipeline 經歷十幾次 LLM 呼叫與多次 web search。

但**它們的核心結構是同一個**：一個 LangGraph `StateGraph`、幾個 node、幾條（條件）邊、一個 state 物件、一個 `compile()`。AI-Q 的厲害不是它「換了套技術」，而是它**把 LangGraph 這個極簡的 primitive 推到極致**——加 escalation、加 HITL、加 checkpoint、加多 LLM 分工——但**外面看起來，YAML 還是那份 YAML**。

這就是 NAT「以 YAML 為核心」這個設計品味的回報：你從寫單一 react_agent，到寫自己的 router，到讀懂 AI-Q 這種等級的編排器，**沒有任何概念上的斷層**，全部都是同一套東西在放大。


<a id="serve"></a>
## 3.4) Server 模式：用 `nat serve` 把 workflow 變成 HTTP API

到目前為止我們都是用 `nat run` 一次跑一個 query 然後就結束——適合開發跟驗證，但要給前端或別的服務呼叫就不對味了。NAT 內建了 FastAPI front-end，**完全不用改 config**，直接 `nat serve` 就會把當前 workflow 開成 HTTP server（預設 `localhost:8000`），同時暴露幾個現成端點：

| 端點 | 用途 |
| --- | --- |
| `POST /chat` | OpenAI Chat Completions 相容（legacy 別名），最容易上手 |
| `POST /v1/chat/completions` | OpenAI Chat Completions 完整路徑 |
| `POST /generate` | 直接呼叫 workflow function 的純字串輸入 |
| `WS /websocket` | WebSocket 雙向串流 |

（背後機制：`nat serve` 其實就是 `nat start fastapi` 的別名；當 config 沒有 `general.front_end:` 區塊時，NAT 會自動套用 `FastApiFrontEndConfig` 的預設值。所以前面我們寫的 `config_simple_researcher.yml` 不用任何修改就能直接拿來 serve。）

<span style="color:rgb(0, 31, 153); font-style: italic;">想客製 port、host 或新增自訂端點時，才需要在 config 的 `general:` 底下加 `front_end: { _type: fastapi, port: 8080, ... }`——但對這份 notebook 不需要。</span>

**跟 AI-Q Web UI 部署的關係**：下節 Docker 部署裡 `aiq-agent` 容器在做的事跟這裡本質上是同一件——把 workflow 包成 HTTP 端點給前端呼叫。差別只是 AI-Q 換上更進階的 `aiq_api` front-end，多了 async jobs + SSE 串流；底下的 NAT runtime 跟 YAML 都沒變。


In [11]:
%%bash --bg
nat serve --config_file config_simple_researcher.yml

server 啟動需要幾秒鐘。下面這個 cell 會先 `sleep` 等它起來，再對預設的 `/chat` 端點送一題 meta query——**payload 是 OpenAI Chat Completions 相容格式**（跟你呼叫 NIM、ChatGPT API 一模一樣，所以前端可以直接拿 OpenAI 的 SDK 來打）：


In [14]:
%%bash
curl --request POST \
  --url http://localhost:8000/chat \
  --header 'Content-Type: application/json' \
  --data '{
    "messages": [
      {"role": "user", "content": "NVIDIA Spectrum-X 主要是設計來做什麼用的？"}
    ]
  }' | jq

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2352  100  2231  100   121    207     11  0:00:11  0:00:10  0:00:01   627


{
  "id": "research_response",
  "object": "chat.completion",
  "model": "chat_deepresearcher_agent",
  "created": 1779157442,
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "NVIDIA Spectrum‑X 主要是一套 **專為 AI 雲端打造的以以太網為基礎的全端網路平台**，其核心設計目標是：\n\n* **加速 AI 工作負載的通訊**：提供高效、低延遲的以太網連線，讓大規模 AI 訓練與推論能在多個 GPU 叢集之間快速交換資料，整體效能可提升約 **1.6 倍**[1]。 \n* **提升 AI 網路效率**：結合 **擴散控制、自適應路由與 AI 驅動的遙測** 等先進功能，使網路資源能更智慧地分配與管理，降低瓶頸，提高整體吞吐量[2]。 \n* **支援全端堆疊**：從 GPU、CPU 到網路層的完整 NVIDIA 全端解決方案（包括 NVLink、GPU‑Direct 等）緊密整合，讓 AI 叢集的擴展與佈建更加順暢[3]。 \n* **適用於大規模 AI 雲端**：被 Meta、Oracle 等業界領袖採用，用於加速其內部的 AI 訓練與推論基礎設施，證明了 Spectrum‑X 在實際大規模 AI 工作負載中的可靠性與效能[4]。\n\n簡單來說，NVIDIA Spectrum‑X 的主要用途是 **提升 AI 計算與網路通訊的效能與擴展性，讓大規模 AI 訓練與推論能在資料中心內更快、更有效率地運作**[1][3]。\n\n**References:**  \n- [1] NVIDIA Spectrum-X Ethernet Platform for Giga-Scale AI - https://www.nvidia.com/en-us/networking/spectrumx/  \n- [2] NVIDIA Spectrum-X Ethernet Switches Speed Up Networks for Meta and Oracle | 

你會拿到跟 `nat run` 同一條 meta 分支的回答，只是這次是透過 HTTP 回來。要繼續對話只要在 `messages` 陣列裡多塞幾輪 user / assistant 訊息即可；要切換 shallow / deep query 也只是改 `content` 字串。

拿到回覆後**記得把背景的 `nat serve` 程序關掉**，避免 port 8000 被佔住：


In [33]:
!pkill -9 -f "nat serve"

<a id="deploy"></a>
# 4.0) 認識 AI-Q 的 Web UI 部署架構

CLI 模式適合開發與快速驗證，但 AI-Q 真正設計給終端使用者用的是 **Web UI**。

<span style="color:rgb(0, 31, 153); font-style: italic;">本節是「概念導覽」——不在 notebook 裡實際拉容器（build + pull 大概要幾分鐘）；想自己跑的話照下一個 cell 的指令貼到 terminal 即可。</span>

**整體架構：三個容器，一份 Docker Compose**

```
 ┌────────────────────┐
 │ aiq-blueprint-ui   │  ← React 前端，port 3000
 │  (Web UI)          │     使用者開瀏覽器看到的就是它
 └─────────┬──────────┘
           │ HTTP / SSE
 ┌─────────▼──────────┐
 │ aiq-agent          │  ← 後端 API server，跑 `nat serve`
 │  (NAT runtime)     │     裡面就是前面 CLI 跑的同一份 workflow
 └─────────┬──────────┘
           │ checkpoints
 ┌─────────▼──────────┐
 │ aiq-postgres       │  ← PostgreSQL，存 LangGraph 的 checkpoint
 │  (State store)     │     做 HITL / 長對話需要的狀態保存
 └────────────────────┘
```

三個容器透過 `deploy/compose/docker-compose.yaml` 一次拉起來。前後端隔離乾淨：UI 換版不影響 agent，agent 換 workflow 也不用動 UI。

**重點觀念**：`aiq-agent` 容器裡跑的就是 `nat serve`——跟前面 CLI 用的 `nat run` 是**同一個 NAT runtime、同一份 YAML、同一個 workflow**。換句話說，Web UI 部署不是「另一套東西」，只是把 CLI 那層換成 HTTP/SSE 給前端用。


**部署流程（三步驟，這份 notebook 不實際執行，貼到 terminal 即可）**

**步驟 1：準備 `.env` 給 Docker Compose 讀取**

```bash
cat > deploy/.env <<EOF
APP_ENV=development
LOG_LEVEL=INFO
AIQ_DEV_ENV=cli
NVIDIA_API_KEY=${NVIDIA_API_KEY}
TAVILY_API_KEY=${TAVILY_API_KEY}
EOF
```

**步驟 2：build image、背景啟動三個容器**

```bash
docker compose -f deploy/compose/docker-compose.yaml build
docker compose -f deploy/compose/docker-compose.yaml up -d
```

第一次 build 大約要 5 分鐘（會編好 `aiq-agent` 與 `aiq-blueprint-ui` 兩個 image），再加上拉一個 `postgres:16-alpine`。

**步驟 3：確認三個容器都跑起來**

```bash
docker ps --format 'table {{.ID}}\t{{.Names}}\t{{.Status}}'
```

預期看到：

```
CONTAINER ID   NAMES                STATUS
...            aiq-blueprint-ui     Up X minutes (healthy)
...            aiq-agent            Up X minutes
...            aiq-postgres         Up X minutes (healthy)
```

**體驗 Web UI**

打開瀏覽器到 [http://localhost:3000](http://localhost:3000)，會看到 AI-Q 的對話介面：

- **右側 Data Sources 面板**：開關 Web Search、Paper Search、Knowledge Layer 等資料來源
- **輸入框送出 query 之後**：agent 會走完整的 LangGraph workflow，深度研究階段透過 SSE 把進度（規劃章節 → 蒐證 → 彙整引用）即時推給前端

用完關掉容器：

```bash
docker compose -f deploy/compose/docker-compose.yaml down -v
```


<a id="build-your-own"></a>
# 4.5) 動手 10 分鐘：打造你自己的 Agent 與工具

整份 notebook 看下來，AI-Q 那麼複雜的編排其實底層就是 **LLM + 工具 + prompt + YAML** 的組合放大。最後我們花 10 分鐘讓你親手把這個 pattern 兜一遍——你做出來的東西**結構上就是 AI-Q `shallow_research_agent` 的化簡版**。

**任務**：用你在前一份 `adding_tools_to_agents.ipynb` 學過的流程，from scratch 兜出一個能查 NVIDIA GPU 規格的 `react_agent`。

| Task | 你要做的事 |
| --- | --- |
| **① 包工具** | 把下一個 cell 給你的原始 Python function `lookup_nvidia_gpu_spec` 包成 NAT function（`@register_function` + `FunctionBaseConfig` + `FunctionInfo.from_fn` 三件套）|
| **② 寫 config** | 自己手寫 `config.yml`（不給完整模板，只給下方 cheat sheet 的關鍵值）|
| **③ 跑起來** | reinstall + `nat run` 驗證能查到 H100 / A100 / B200 / GB200 的規格、領域外問題會被 agent 推回 |

---

### Cheat sheet（你寫 `config.yml` 會用到的值）

**`llms:` 區**（用 nemotron，搭配 ReAct 友善的 hyperparams）

| 欄位 | 值 |
| --- | --- |
| `_type` | `nim` |
| `model_name` | `nvidia/nemotron-3-nano-30b-a3b` |
| `base_url` | `https://integrate.api.nvidia.com/v1` |
| `temperature` | `0.1` |
| `top_p` | `0.3` |
| `max_tokens` | `16384` |
| `chat_template_kwargs.enable_thinking` | `false`（關掉 nemotron 的 `<think>` 區塊避免跟 ReAct parser 打架）|

**`workflow:` 區**

| 欄位 | 值 |
| --- | --- |
| `_type` | `react_agent` |
| `llm_name` | 你在 `llms:` 區自己命名的 key |
| `tool_names` | 你在 `functions:` 區自己命名的 key（list 形式）|
| `system_prompt` | 寫個 NVIDIA 產品助理的人設（文末 `<details>` 有 3 個範例）|
| `verbose` | `true`（才看得到 agent 與工具的對話）|
| `retry_parsing_errors` | `true` |

**`functions:` 區**：自己命名一個 key，`_type` 對應到你在 `FunctionBaseConfig(name=...)` 寫的名字。

---

### 檔案結構（`nat workflow create` 會幫你建好）

<pre>
my_lab_agent/
  src/my_lab_agent/
    register.py         # 在這裡加 `from . import 你的工具檔`
    gpu_spec_tool.py    # 你寫的 tool（task ①）
  configs/
    config.yml          # 你寫的 config（task ②）
</pre>

**驗收**：跑得起來、agent 能透過你的工具查到 GPU 規格、領域外問題（例如「週末會下雨嗎」）會被推回 NVIDIA 主題。

<span style="color:rgb(0, 31, 153); font-style: italic;">卡關的話：文末 `<details>` 收了 tool wrapping 與 config.yml 的完整參考解，可以對照——但建議先自己做一次再看。</span>


In [ ]:
# ★ Task ① 的素材：下面是一個原始 Python function ★
# 你的工作：照前一份 adding_tools_to_agents 學的 pattern，
# 把它包成 NAT function（寫進 my_lab_agent/src/my_lab_agent/gpu_spec_tool.py）。

def lookup_nvidia_gpu_spec(gpu_model: str) -> str:
    """Look up datasheet-level specs for a NVIDIA GPU model name.

    Returns a one-line summary including architecture, year, memory,
    bandwidth, TDP, interconnect, and process node. Returns a helpful
    error message if the model is unknown.
    """
    SPEC_SHEET = {
        "h100":  {"arch": "Hopper",            "year": 2022,
                  "mem": "80 GB HBM3",          "bw": "3.35 TB/s",
                  "tdp": "700W",   "interconnect": "NVLink 4.0 (900 GB/s)",
                  "process": "TSMC 4N"},
        "a100":  {"arch": "Ampere",            "year": 2020,
                  "mem": "80 GB HBM2e",         "bw": "2.04 TB/s",
                  "tdp": "400W",   "interconnect": "NVLink 3.0 (600 GB/s)",
                  "process": "TSMC 7nm"},
        "b200":  {"arch": "Blackwell",         "year": 2024,
                  "mem": "192 GB HBM3e",        "bw": "8 TB/s",
                  "tdp": "1000W",  "interconnect": "NVLink 5.0 (1.8 TB/s)",
                  "process": "TSMC 4NP"},
        "gb200": {"arch": "Blackwell + Grace", "year": 2024,
                  "mem": "384 GB HBM3e (2-GPU superchip)",
                  "bw": "16 TB/s", "tdp": "2700W",
                  "interconnect": "NVLink 5.0 (1.8 TB/s)",
                  "process": "TSMC 4NP"},
    }
    key = gpu_model.lower().strip()
    if key not in SPEC_SHEET:
        available = ", ".join(sorted(SPEC_SHEET.keys()))
        return f"找不到型號 '{model}'。目前可查詢：{available}"
    s = SPEC_SHEET[key]
    return (
        f"{gpu_model.upper()} ({s['arch']}, {s['year']}): "
        f"記憶體 {s['mem']}（頻寬 {s['bw']}）、TDP {s['tdp']}、"
        f"互連 {s['interconnect']}、製程 {s['process']}"
    )


# 先在本 cell 直接呼叫一下確認 raw function 行為（再去包成 NAT function）
print(lookup_nvidia_gpu_spec("h100"))
print(lookup_nvidia_gpu_spec("xyz"))

In [ ]:
# 用前一份 notebook 學過的指令，幫你的 lab agent 建好整包 scaffolding
!nat workflow create my_lab_agent

In [ ]:
%%writefile my_lab_agent/src/my_lab_agent/gpu_spec_tool.py
# ★ Task ①：把上一個 cell 的 raw function 包成 NAT function ★
#
# 提示：照 adding_tools_to_agents notebook 學的「Config + @register_function
# + 內部 async def + FunctionInfo.from_fn」四件套。
#
# 已經幫你擺好骨架，你要做的：
#   - TODO ① 給 FunctionBaseConfig 一個唯一的 name（這個 name 就是
#            你寫 config.yml 時的 _type）
#   - TODO ② 把上面那個 SPEC_SHEET 跟查詢邏輯，搬進下面 async def 內
#   - TODO ③ 用 FunctionInfo.from_fn 把 _lookup yield 出去

from nat.builder.builder import Builder
from nat.builder.framework_enum import LLMFrameworkEnum
from nat.builder.function_info import FunctionInfo
from nat.cli.register_workflow import register_function
from nat.data_models.function import FunctionBaseConfig


# TODO ①：給一個有辨識度的 name，這就是之後 config.yml 裡的 _type
class GpuSpecLookupConfig(FunctionBaseConfig, name="..."):
    """Look up NVIDIA GPU specs from a built-in cheat sheet."""
    # 這個 tool 不需要任何 config 參數
    pass


@register_function(
    config_type=GpuSpecLookupConfig,
    framework_wrappers=[LLMFrameworkEnum.LANGCHAIN],
)
async def gpu_spec_lookup_function(config: GpuSpecLookupConfig, _builder: Builder):
    """Look up datasheet-level specs for a NVIDIA GPU model name."""

    # TODO ②： SPEC_SHEET 整個搬進這裡
    SPEC_SHEET = {
        # ... 你的 dict ...
    }

    async def _lookup(model: str) -> str:
        """
        Look up datasheet-level specs for a NVIDIA GPU model name.
        Returns architecture, year, memory, bandwidth, TDP, interconnect,
        and process node. Case-insensitive on the model name.

        Args:
            model: GPU model name (e.g. "H100", "A100", "B200", "GB200")

        Returns:
            One-line spec summary, or an error message listing known models.
        """
        # TODO ②（續）：把查詢邏輯寫在以下區塊
        ...

    # TODO ③：把 _lookup yield 出去（提示：FunctionInfo.from_fn）
    ...

In [ ]:
%%bash
# 把新工具加進 register.py，讓 NAT 載入 package 時會掃到 @register_function
echo "" >> my_lab_agent/src/my_lab_agent/register.py
echo "from . import gpu_spec_tool" >> my_lab_agent/src/my_lab_agent/register.py

# 重新安裝這個 package，讓剛才的改動生效
nat workflow reinstall my_lab_agent

In [ ]:
%%writefile my_lab_agent/configs/config.yml
# ★ Task ②：自己寫整份 config.yml ★
# 對照上方 markdown 的 Cheat Sheet 把值填進去。
# 提醒：functions 區裡的 _type 要對應你在 gpu_spec_tool.py 給
# FunctionBaseConfig 的 name=

llms:
  # TODO：把 cheat sheet 的 LLM 區欄位填進來
  # <你的命名>:
  #   _type: ...
  #   model_name: ...
  #   ...

functions:
  # TODO：把你的 GPU 規格查詢工具掛進來
  # <你的命名>:
  #   _type: <你在 FunctionBaseConfig 寫的 name>

workflow:
  # TODO：用 react_agent，掛上 LLM 與工具，寫好 system_prompt
  # _type: ...
  # llm_name: ...
  # tool_names:
  #   - ...
  # system_prompt: |
  #   你是 NVIDIA 產品技術助理...

In [ ]:
# 領域內那題 —— 預期 agent 會呼叫你的工具，查到 H100 / A100 規格後比較
!nat run --config_file=my_lab_agent/configs/config.yml \
    --input "請告訴我 NVIDIA H100 的記憶體頻寬與 TDP，並跟 A100 比較"

# 領域外那題（測 agent 的「人設不崩」）—— 把下面 # 拿掉跑跑看
# !nat run --config_file=my_lab_agent/configs/config.yml \
#     --input "請問週末台北會下雨嗎？"

<details>
<summary><b>卡關了？點開看「Task ① 工具包裝」參考解</b></summary>

<pre>
from nat.builder.builder import Builder
from nat.builder.framework_enum import LLMFrameworkEnum
from nat.builder.function_info import FunctionInfo
from nat.cli.register_workflow import register_function
from nat.data_models.function import FunctionBaseConfig


class GpuSpecLookupConfig(FunctionBaseConfig, name="gpu_spec_lookup"):
    """Look up NVIDIA GPU specs from a built-in cheat sheet."""
    pass


@register_function(
    config_type=GpuSpecLookupConfig,
    framework_wrappers=[LLMFrameworkEnum.LANGCHAIN],
)
async def gpu_spec_lookup_function(config, _builder):
    SPEC_SHEET = {
        "h100":  {"arch": "Hopper", "year": 2022, "mem": "80 GB HBM3",
                  "bw": "3.35 TB/s", "tdp": "700W",
                  "interconnect": "NVLink 4.0 (900 GB/s)",
                  "process": "TSMC 4N"},
        # ... 其他型號 ...
    }

    async def _lookup(model: str) -> str:
        """Look up datasheet-level specs for a NVIDIA GPU model name.

        Args:
            model: GPU model name (e.g. "H100", "A100", "B200", "GB200")
        """
        key = model.lower().strip()
        if key not in SPEC_SHEET:
            available = ", ".join(sorted(SPEC_SHEET.keys()))
            return f"找不到型號 '{model}'。目前可查詢：{available}"
        s = SPEC_SHEET[key]
        return (f"{model.upper()} ({s['arch']}, {s['year']}): "
                f"記憶體 {s['mem']}（頻寬 {s['bw']}）、TDP {s['tdp']}、"
                f"互連 {s['interconnect']}、製程 {s['process']}")

    yield FunctionInfo.from_fn(_lookup, description=_lookup.__doc__)
</pre>

</details>

<details>
<summary><b>卡關了？點開看「Task ② config.yml」參考解</b></summary>

<pre>
llms:
  agent_llm:
    _type: nim
    model_name: nvidia/nemotron-3-nano-30b-a3b
    base_url: "https://integrate.api.nvidia.com/v1"
    temperature: 0.1
    top_p: 0.3
    max_tokens: 16384
    chat_template_kwargs:
      enable_thinking: false

functions:
  gpu_spec:
    _type: gpu_spec_lookup    # 對應你在 FunctionBaseConfig 寫的 name

workflow:
  _type: react_agent
  llm_name: agent_llm
  tool_names:
    - gpu_spec
  verbose: true
  retry_parsing_errors: true
  system_prompt: |
    你是 NVIDIA 產品技術助理，專門回答 NVIDIA GPU、AI 平台與軟體堆疊
    （CUDA、cuDNN、TensorRT、Triton）的問題。

    回答原則：
    - 一律先用 gpu_spec 工具查 GPU 規格再回答（不要憑記憶推測規格）
    - 引用時標出「根據內部規格表」讓使用者知道資料來源
    - 技術名詞保留英文（例：FP8、HBM3e、NVLink、PCIe Gen5）
    - 若問題不在 NVIDIA 產品範圍內（例如問天氣、跟 GPU/AI 無關的八卦），
      客氣回覆「我專精 NVIDIA 產品相關問題，這題不在我的領域」並
      引導使用者回到 NVIDIA 主題
</pre>

</details>

<details>
<summary><b>想換主題？這裡有 3 個替代 system_prompt 範例（點開）</b></summary>

想做你領域的版本，只要：(a) 改 raw function 改成查你領域的資料、(b) 改 `system_prompt`、(c) `config.yml` 其他不用動。

<b>🧳 旅遊規劃師</b>

<pre>
你是貼心的台日旅遊規劃師。回答原則：
- 用工具查景點/交通/餐廳資訊
- 行程附時間、交通方式、預估費用（日幣 + 台幣換算）
- 優先推「在地人推薦、不在熱門路線上」的選擇
- 結尾附「踩雷提醒」與「省錢小訣竅」
</pre>

<b>📄 履歷顧問</b>

<pre>
你是科技業履歷顧問，專長 SWE / ML / DevOps 履歷。回答原則：
- 用工具查當前職缺的真實 JD 與 skill demand
- 給回饋時用 STAR（Situation-Task-Action-Result）結構評論
- 同時給 3 個改寫範例（保守 / 平衡 / 進取版）
- 結尾附 3 個推薦的 JD 連結作為對標
</pre>

<b>📰 半導體新聞分析師</b>

<pre>
你是半導體產業分析師。回答原則：
- 用工具查最新新聞（24-48 小時內優先）
- 引用優先順序：EE Times、AnandTech、SemiAnalysis、Tom's Hardware
  &gt; 主流新聞
- 結論分「對 AI 訓練/推論的影響」「對台廠供應鏈的影響」兩個面向
- 結尾附 1 個「值得追蹤的訊號」
</pre>

</details>

---

**收尾觀察**：你剛剛親手把一個原始 Python function 包成 NAT tool、自己組裝出 `config.yml`、再用 `react_agent` 把 LLM 跟工具兜起來——**結構上就是 AI-Q `shallow_research_agent` 的化簡版**。把它擴大（多接幾個工具、把它變成 sub-agent、再加一條 LangGraph 條件邊路由），就會自然長成 AI-Q 的樣子。

**你已經成功熟悉NAT的流程了。** 接下來怎麼用，就看下節「下一步」。


<a id="next-steps"></a>
# 5.0) 下一步

走到這裡，你已經完整體驗透過 NeMo Agent Toolkit 搭建的 AI-Q 了：跑過 meta / shallow / deep 三條路徑，看過了 Web UI 部署架構，也親手寫了一個 NAT tool、自己組裝出 `config.yml`、跑起一隻屬於你領域的 react_agent。

**恭喜你完成本次的 NeMo Agent Toolkit 的教學。**

**後續推薦你做這幾件事來更熟悉 NeMo Agent Toolkit 與 AI-Q：**

- 翻一翻 AI-Q 官方的進階 notebook，了解更多進階做法：
    - `docs/notebooks/1_Deep_Researcher_Web_Search.ipynb`：聚焦 Deep Researcher、加上 `nat eval` 評測
    - `docs/notebooks/2_Deep_Researcher_Customization.ipynb`：論文搜尋、各 agent 換用不同 LLM、改 prompt、啟用 Knowledge Layer
- 試著把 `config_simple_researcher.yml` 加上 `clarifier_agent`，體驗深度研究前的 HITL 澄清與計畫核可流程
- 寫一個你自己領域的 NAT function（架構跟 bridge 的 `nvidia_blog_search` 一樣），把它包成獨立套件 `pip install -e` 進 AI-Q，做一個真正屬於你的 domain researcher
- 修改 `src/aiq_agent/agents/deep_researcher/prompts/orchestrator.j2`，給 Deep Researcher 加上你的領域人設與引用偏好
- 看 `docs/source/architecture/` 底下的文件，深入了解每個 agent 的內部流程與 prompt 設計
- 如果要進 production，研究 `deploy/helm/` 與 `deploy/compose/` 兩種部署方式，並評估是否要加 OAuth 與 Foundational RAG

祝你玩得愉快！
